# Inference pipeline: paragraph -> linked entities

Takes any plain-text paragraph (paste it in the input cell below and run all
cells) and returns the dataframe from the assignment (Mention, Candidate
concept, Wikidata ID, Confidence), plus the structured JSON output
(alternatives + evidence) per mention.

No new logic is implemented here - every phase calls the pipeline methods:
1. term extraction: `src/text_preprocess.py` (spaCy sm NER + d1 pass)
2. candidate generation: `src/candidate_generator.py`
3. document-level graph: `src/ranker/utils.py` (`graphs_from_rows`)
4. ranker: method v2 (`src/ranker/cr_v2.py`) - semantic similarity with
   document-level graph context; the only place the ranker is called
5.-6. output assembly: dataframe + structured JSON (no scoring)

Requires Ollama running locally with `embeddinggemma` and `d1-omni:600m`.

cr_v2.py was selected as the final ranker, as it was having highest accuracy of 78.3%

In [2]:
import contextlib
import io
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
sys.path += [str(ROOT / "src"), str(ROOT / "src/ranker")]

from text_preprocess import nlp_sm as nlp, text_preprocess, covered
from candidate_generator import candidate_generator
from build_candidate_store import candidates_as_json

# note: utils/cr_v2 have to be imported flat (src/ranker on sys.path), same way
# the ranker modules import each other. "from ranker import utils" would load a
# second utils module, and the graph seeded there in step 3 would be invisible
# to cr_v2 - it would silently rank without graph context (so basically v1).
import ranker.utils
from ranker.utils import graphs_from_rows, print_graph, to_json
from ranker.cr_v2 import rank

## 0. Input text

paste any new unseen paragraph here - everything downstream runs off this one
variable.

In [3]:
paragraph = ("Marie Curie conducted research on radioactivity in Paris "
             "and won the Nobel Prize in Physics.")

## 1. Term extraction

`text_preprocess` on the paragraph gives the merged NER entities (spaCy + d1)
and the noun phrases. noun phrases already `covered` by NER are dropped, the
rest (e.g. "radioactivity") become mentions too. each mention then goes through
`text_preprocess` again with `mention=` set, exactly how the candidate stores
were built: that gives its parent sentence (the ranking context) and the
expanded query forms for retrieval (leading determiners stripped, wordnet
synonyms only for mentions NER did not cover).

In [4]:
doc = nlp(paragraph)
pre = text_preprocess(doc)
ents = {t.lower().strip() for t, _ in pre["ner_entities"]}
terms = [t for t, _ in pre["ner_entities"]]
terms += [np for np in pre["noun_phrases"]
          if not covered(np.lower().strip(), ents)]

mentions, seen = [], set()
for m in (t.strip() for t in terms):
    if not m or m.lower() in seen:
        continue
    seen.add(m.lower())
    p = text_preprocess(doc, mention=m)
    mentions.append((m, p["parent_sentence"] or paragraph, p))

print("mentions:", [m for m, _, _ in mentions])

mentions: ['Marie Curie', 'Paris', 'the Nobel Prize in Physics', 'research', 'radioactivity']


## 2. Candidate generation phase

`candidate_generator` per mention, reusing the mention's preprocessing
(expanded query forms) from step 1. `candidates_as_json` converts the output
to the same store format the rankers consume. no ranking happens here - the
candidates are just collected per mention for the two phases below.

In [5]:
cands_by_mention = {}
for m, sent, p in mentions:
    with contextlib.redirect_stdout(io.StringIO()):  # generator is chatty
        result = candidate_generator(m, sent, preprocessed=p)
    cands_by_mention[(m, sent)] = candidates_as_json(result)
    print(f"{m!r}: {len(cands_by_mention[(m, sent)])} candidates")

temp = temp =   {'mention_form': 'marie curie', 'mention_type': 'surface', 'QID': 'Q7186', 'label': 'Marie Curie', 'linked_by_alias': 0, 'linked_by_label': 1, 'rank_by_wiki': 1, 'is_disambiguation_page': 0, 'has_description': 1}
temp =  {'mention_form': 'marie curie', 'mention_type': 'surface', 'QID': 'Q114939443', 'label': 'Marie Curie', 'linked_by_alias': 0, 'linked_by_label': 1, 'rank_by_wiki': 2, 'is_disambiguation_page': 0, 'has_description': 1}
temp =  {'mention_form': 'marie curie', 'mention_type': 'surface', 'QID': 'Q30668387', 'label': 'Marie-Curie', 'linked_by_alias': 0, 'linked_by_label': 1, 'rank_by_wiki': 3, 'is_disambiguation_page': 0, 'has_description': 1}
temp =  {'mention_form': 'marie curie', 'mention_type': 'surface', 'QID': 'Q134450801', 'label': 'Marie Curie', 'linked_by_alias': 0, 'linked_by_label': 1, 'rank_by_wiki': 4, 'is_disambiguation_page': 0, 'has_description': 1}
temp =  {'mention_form': 'marie curie', 'mention_type': 'surface', 'QID': 'Q1613719', 'label':

## 3. Document-level graph phase

`utils.graphs_from_rows` (the same graph construction the rankers use on the
candidate store) builds the document level graph on the fly from the in-memory
candidates. the whole paragraph is one group, like a doc_id in the AIDA store.
seeding `utils._graphs` with it is all v2 needs: `cr_v2.rank` calls
`graph_candidate_text` and `graph_bonus`, which get the graph through
`utils.candidate_graph`, and that reads this cache. still no ranking here -
this only prepares the context the ranker reads in step 4.

In [6]:
from ranker import utils

utils._graphs = graphs_from_rows(
    [(m, sent, "doc_0", cands) for (m, sent), cands in cands_by_mention.items()],
    doc_level=True)
print_graph(utils._graphs[0]["doc_0"][0])

Graph with 1062 nodes and 1109 edges
╟── Siege of Paris
╎   └── Paris ─ Marie Curie, Portal:Paris, Category:People from Paris, mayor of Paris, Prévôt des marchands de Paris, Genevieve of Paris, France, Kingdom of France, Francia, German military administration in occupied France during World War II, First French Empire, French First Republic, Île-de-France, Second French Empire, Neustria, Bourbon Restoration in France, metropolitan France, Seine, Grand Paris, French Fourth Republic, French Second Republic, Paris Commune, zone de défense et de sécurité de Paris, French colonial empire, arrondissement of Paris, Paris Department, Vichy France, Parisian cuisine, history of Paris, Seine, Bassin de la Villette, Canal Saint-Martin, Canal de l'Ourcq, list of monuments and sites in Paris, list of historic monuments in Paris, Rome, Tokyo, Kyoto, Berlin, Ramallah, Seoul, Cairo, Chicago, Torreón, San Francisco, Kyiv, Washington, D.C., Marrakech, Porto Alegre, Dubai, Beijing, Mexico City, Saint Pet

## 4. Ranker phase (cr_v2)

**this is the only place the ranker is called**: one
`rank(mention, sentence, candidates)` call per mention (`src/ranker/cr_v2.py`).
it scores each candidate with embedding cosine over the graph-verbalized
evidence + lexical priors + the graph coherence bonus (reading the document
graph seeded in step 3) and returns the candidates sorted best-first. nothing
else happens in this cell.

In [7]:
ranked_by_mention = {}
for (m, sent), cands in cands_by_mention.items():
    ranked_by_mention[(m, sent)] = rank(m, sent, cands)  # <-- the ranker call
    print(f"{m!r}: ranked {len(cands)} candidates")

'Marie Curie': ranked 7 candidates
'Paris': ranked 12 candidates
'the Nobel Prize in Physics': ranked 7 candidates
'research': ranked 11 candidates
'radioactivity': ranked 14 candidates


## 5. Assignment dataframe

no scoring here - `to_json` just turns each mention's already-ranked list from
step 4 into the structured record (selected + alternatives with softmax
confidence), and the dataframe keeps the top candidate per mention.

In [8]:
rows, outputs = [], []
for (m, sent), ranked in ranked_by_mention.items():
    out = to_json(m, sent, ranked)
    outputs.append(out)
    sel = out["selected"]
    rows.append({"Mention": m,
                 "Candidate concept": sel["concept"] if sel else None,
                 "Wikidata ID": sel["qid"] if sel else None,
                 "Confidence": sel["confidence"] if sel else None})

df = pd.DataFrame(rows)
df

,Mention,Candidate concept,Wikidata ID,Confidence
0,Marie Curie,Marie Curie,Q7186,0.1963
1,Paris,Paris,Q90,0.1105
2,the Nobel Prize in Physics,Nobel Prize in Physics,Q38104,0.1755
3,research,research,Q42240,0.1252
4,radioactivity,radioactivity,Q11448,0.0968


## 6. Structured JSON output

per mention: selected concept + QID with confidence, ranked alternatives and
the text evidence each score was computed on (same format the evaluation
writes to `linking_results_cr_v2.json`). results are stored in
`ranker_results/`.

In [9]:
print(json.dumps(outputs[0], indent=2, ensure_ascii=False)[:2000])

{
  "mention": "Marie Curie",
  "sentence": "Marie Curie conducted research on radioactivity in Paris and won the Nobel Prize in Physics.",
  "selected": {
    "concept": "Marie Curie",
    "qid": "Q7186",
    "rank": 1,
    "score": 0.7648,
    "confidence": 0.1963,
    "evidence": "Marie Curie: Polish-born French physicist and chemist (1867–1934)"
  },
  "alternatives": [
    {
      "concept": "Marie Curie",
      "qid": "Q16997351",
      "rank": 2,
      "score": 0.4846,
      "confidence": 0.1483,
      "evidence": "Marie Curie: charity in the UK (also known as Marie Curie Cancer Care)"
    },
    {
      "concept": "Marie Curie",
      "qid": "Q114939443",
      "rank": 3,
      "score": 0.4833,
      "confidence": 0.1481,
      "evidence": "Marie Curie: 2022 Czech book edition"
    },
    {
      "concept": "Marie Curie",
      "qid": "Q134450801",
      "rank": 4,
      "score": 0.4803,
      "confidence": 0.1477,
      "evidence": "Marie Curie: Ferry built in 2019 (also known

In [10]:
out_path = Path("ranker_results/inference_results.json")
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(outputs, indent=2, ensure_ascii=False))
print(f"wrote {out_path}")

wrote ranker_results/inference_results.json
